# Observing Blocks: Spanned parameter space

Owner: **Chris Suberlak** <br>
Last Verified to Run: **2025-12-01** <br>


This notebook allows to plot the parameter space occupied by observing parameters for a chosen science program. The date range and BLOCK name can be updated on the left panel.

In [ ]:
# Times Square Parameters
day_obs_min = 20250828
day_obs_max = 20251201
science_program = 'BLOCK-T644'

In [ ]:
from lsst.summit.utils import ConsDbClient
import os
import numpy as np

# Only modify no_proxy if it exists (RSP environment)
if "no_proxy" in os.environ:
    os.environ["no_proxy"] += ",.consdb"

consdb_url = "http://consdb-pq.consdb:8080/consdb"
client = ConsDbClient(consdb_url)

In [ ]:
print(f'Currently querying consDB for {science_program} for date range {day_obs_min} - {day_obs_max}')

If you would like to choose a different science program, search for a desired phrase, below, and copy-paste the desired block name (eg. "BLOCK-T123") into "science_program" field on the left, and click "update". This will recompute the notebook for that science program.

In [ ]:
from IPython.display import HTML, display
import json
import numpy as np
import random

# Generate unique ID to avoid conflicts
unique_id = random.randint(10000, 99999)

# Query the data
query = f'''
SELECT 
  t.day_obs, 
  t.science_program,
  t.observation_reason
FROM 
  cdb_lsstcam.visit1 AS t
WHERE  
  t.day_obs > {day_obs_min} AND
  t.day_obs < {day_obs_max}
'''

df = client.query(query)
unique_programs = np.unique(df['science_program'])

# Build structured data for JavaScript
programs_data = []
for program in unique_programs:
    mask = df['science_program'] == program
    program_rows = df[mask]
    unique_reasons = np.unique(program_rows['observation_reason'])
    
    programs_data.append({
        'program': str(program),
        'reasons': [str(r) for r in unique_reasons],
        'count': int(len(program_rows))
    })

# Create HTML with inline JavaScript
html_inline = f"""
<div style="max-width: 1200px; margin: 20px auto;">
    <div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); 
                color: white; padding: 20px; border-radius: 8px;">
        <h2 style="margin: 0 0 8px 0;">📋 Available Science Programs</h2>
        <p style="margin: 0; opacity: 0.95;">Date range: {day_obs_min} to {day_obs_max}</p>
        <button id="toggle-btn-{unique_id}" 
                style="margin-top: 15px; padding: 10px 20px; background: white; color: #667eea; 
                       border: none; border-radius: 4px; font-size: 14px; font-weight: bold; 
                       cursor: pointer; box-shadow: 0 2px 4px rgba(0,0,0,0.2);
                       transition: all 0.3s ease;">
            Show Available Programs
        </button>
    </div>
    
    <div id="programs-container-{unique_id}" style="display: none;">
        <div style="background: white; padding: 15px; border-bottom: 1px solid #dee2e6;">
            <input type="text" id="search-input-{unique_id}" 
                   placeholder="Search programs or observation reasons..." 
                   style="width: 100%; padding: 10px; border: 2px solid #ced4da; 
                          border-radius: 4px; font-size: 14px; box-sizing: border-box;">
            <div id="status-{unique_id}" style="margin-top: 8px; padding: 8px; border-radius: 4px; 
                                           background: #d1ecf1; color: #0c5460; font-size: 13px;">
                💡 Start typing to filter
            </div>
        </div>
        
        <div id="results-{unique_id}" style="background: white; padding: 20px; max-height: 600px; 
                                        overflow-y: auto; border-radius: 0 0 8px 8px; box-shadow: 0 2px 8px rgba(0,0,0,0.1);">
        </div>
    </div>
</div>

<script>
(function() {{
    const programsData = {json.dumps(programs_data)};
    const container = document.getElementById('programs-container-{unique_id}');
    const toggleBtn = document.getElementById('toggle-btn-{unique_id}');
    const resultsContainer = document.getElementById('results-{unique_id}');
    const searchInput = document.getElementById('search-input-{unique_id}');
    const statusDiv = document.getElementById('status-{unique_id}');
    
    let initialized = false;
    
    console.log('Programs widget initialized', toggleBtn, container);
    
    // Toggle visibility
    toggleBtn.addEventListener('click', function() {{
        console.log('Button clicked, current display:', container.style.display);
        if (container.style.display === 'none' || container.style.display === '') {{
            container.style.display = 'block';
            toggleBtn.textContent = 'Hide Available Programs';
            toggleBtn.style.background = '#f8f9fa';
            
            // Initialize data on first show
            if (!initialized) {{
                console.log('Initializing programs...');
                initializePrograms();
                initialized = true;
            }}
        }} else {{
            container.style.display = 'none';
            toggleBtn.textContent = 'Show Available Programs';
            toggleBtn.style.background = 'white';
        }}
    }});
    
    function initializePrograms() {{
        programsData.forEach(function(prog) {{
            const div = document.createElement('div');
            div.className = 'prog-block-{unique_id}';
            div.dataset.name = prog.program.toLowerCase();
            
            const reasonsHtml = prog.reasons.map(function(r) {{
                return '<div class="reason-item-{unique_id}" data-reason="' + r.toLowerCase() + '" ' +
                       'style="padding: 6px 10px; margin: 3px 0; background: #f8f9fa; ' +
                       'border-left: 3px solid #667eea; border-radius: 3px; font-size: 13px;">' +
                       r + '</div>';
            }}).join('');
            
            div.innerHTML = 
                '<div style="border: 1px solid #dee2e6; border-radius: 6px; margin-bottom: 15px; overflow: hidden;">' +
                    '<div style="background: #f8f9fa; padding: 12px 15px; border-bottom: 1px solid #dee2e6;">' +
                        '<strong style="font-size: 16px; color: #495057;">' + prog.program + '</strong>' +
                        '<span style="color: #6c757d; margin-left: 10px; font-size: 13px;">' + prog.count + ' visits</span>' +
                    '</div>' +
                    '<div style="padding: 10px 15px;">' + reasonsHtml + '</div>' +
                '</div>';
            
            resultsContainer.appendChild(div);
        }});
        console.log('Initialized', programsData.length, 'programs');
    }}
    
    searchInput.addEventListener('input', function() {{
        const term = searchInput.value.toLowerCase().trim();
        const blocks = resultsContainer.querySelectorAll('.prog-block-{unique_id}');
        
        if (!term) {{
            blocks.forEach(function(b) {{
                b.style.display = 'block';
                const reasons = b.querySelectorAll('.reason-item-{unique_id}');
                reasons.forEach(function(r) {{
                    r.style.display = 'block';
                    r.style.background = '#f8f9fa';
                }});
            }});
            statusDiv.textContent = '💡 Start typing to filter';
            statusDiv.style.background = '#d1ecf1';
            statusDiv.style.color = '#0c5460';
            return;
        }}
        
        let matched = 0;
        blocks.forEach(function(block) {{
            const name = block.dataset.name;
            const nameMatch = name.includes(term);
            const reasons = block.querySelectorAll('.reason-item-{unique_id}');
            let hasMatch = nameMatch;
            
            reasons.forEach(function(r) {{
                const reasonMatch = r.dataset.reason.includes(term);
                if (reasonMatch) {{
                    r.style.display = 'block';
                    r.style.background = '#e3f2fd';
                    hasMatch = true;
                }} else if (nameMatch) {{
                    r.style.display = 'block';
                    r.style.background = '#f8f9fa';
                }} else {{
                    r.style.display = 'none';
                }}
            }});
            
            block.style.display = hasMatch ? 'block' : 'none';
            if (hasMatch) matched++;
        }});
        
        statusDiv.textContent = matched > 0 ? '✓ Found ' + matched + ' program(s)' : '⚠️ No matches';
        statusDiv.style.background = matched > 0 ? '#d4edda' : '#fff3cd';
        statusDiv.style.color = matched > 0 ? '#155724' : '#856404';
    }});
}})();
</script>
"""

display(HTML(html_inline))

print(f"✓ Ready to display {len(programs_data)} science programs")
print(f"Date range: {day_obs_min} to {day_obs_max}")
print("Click 'Show Available Programs' to view the list")

In [ ]:
from lsst.summit.utils import ConsDbClient
import matplotlib.pyplot as plt 
from matplotlib.gridspec import GridSpec

consdb_url = "http://consdb-pq.consdb:8080/consdb"
client = ConsDbClient(consdb_url)

query = f'''
SELECT 
  t.visit_id, t.day_obs, 
  t.seq_num, t.physical_filter,
  t.azimuth, t.altitude, t.airmass, t.science_program,
  t.observation_reason, t.dimm_seeing,
  q.physical_rotator_angle,
  q.aos_fwhm, q.psf_sigma_median, q.donut_blur_fwhm,
  q.ringss_seeing
FROM 
  cdb_lsstcam.visit1 AS t,
  cdb_lsstcam.visit1_quicklook AS q
WHERE 
  t.visit_id = q.visit_id 
  AND t.day_obs > {day_obs_min} 
  AND t.day_obs < {day_obs_max} 
  AND t.science_program = '{science_program}'
'''
df = client.query(query)
max_day_obs = max(np.unique(df['day_obs']))

In [ ]:
from bokeh.plotting import figure, output_file, save, show, output_notebook
from bokeh.layouts import gridplot, column, row
from bokeh.models import HoverTool, ColumnDataSource, Div, Slider, CustomJS, TextInput, Range1d
import numpy as np
from datetime import datetime

# Enable notebook output
output_notebook()

# Your band colors
band_colors = {
    "u": "#0c71ff",
    "g": "#49be61",
    "r": "#c61c00",
    "i": "#ffc200",
    "z": "#f341a2",
    "y": "#5d0000",
}

# Variables to plot
variables = ['altitude', 'airmass', 
             'dimm_seeing', 'physical_rotator_angle',
             'aos_fwhm', 'psf_fwhm_asec', 'donut_blur_fwhm']

labels = ['Altitude', 'Airmass', 'DIMM Seeing', 'Rotator',
          'AOS FWHM', 'PSF FWHM', 'Donut Blur']

# Calculate PSF FWHM from psf_sigma_median
SIGMA2FWHM = np.sqrt(8 * np.log(2))
pixToArcseconds = 0.199225 # nominal plate scale 
df['psf_fwhm_asec'] = df["psf_sigma_median"].astype(float) * SIGMA2FWHM * pixToArcseconds

# **KEY FIX: Create mask for rows with valid data in all variables**
mask = np.ones(len(df), dtype=bool)
for var in variables:
    # Check for masked values, NaN, or None
    col_data = df[var]
    if hasattr(col_data, 'mask'):
        mask &= ~col_data.mask
    mask &= ~np.isnan(col_data.astype(float, copy=False))

df_clean = df[mask]

# Convert to pandas for easier handling
df_clean_pd = df_clean.to_pandas()

# Extract bands and colors
bands = [pf.split('_')[0] for pf in df_clean['physical_filter']]
colors = [band_colors[band] for band in bands]

n_vars = len(variables)

# Add title
if science_program == 'BLOCK-T614':
    test_title = 'open loop aos stability test'
elif science_program == 'BLOCK-T641':
    test_title = 'closed loop aos stability test'
elif science_program == 'BLOCK-T635':
    test_title = 'focus sweep test'
else:
    test_title = np.unique(df['observation_reason'])[0]

# Prepare data
df_pd = df_clean_pd.copy()
df_pd['band'] = bands
df_pd['color'] = colors
df_pd['day_obs'] = df_clean['day_obs'].astype(int)

# Get ONLY available dates from the data (sorted)
available_dates = sorted(df_pd['day_obs'].unique())
n_dates = len(available_dates)

print(f"Available dates in dataset: {n_dates} unique dates")
print(f"Date range: {available_dates[0]} to {available_dates[-1]}")

# Find indices for initial day_obs values
try:
    start_idx = available_dates.index(day_obs_min) if day_obs_min in available_dates else 0
except ValueError:
    start_idx = 0

try:
    end_idx = available_dates.index(day_obs_max) if day_obs_max in available_dates else n_dates - 1
except ValueError:
    end_idx = n_dates - 1

# Create initial filtered data
initial_mask = (df_pd['day_obs'] >= available_dates[start_idx]) & (df_pd['day_obs'] <= available_dates[end_idx])
df_filtered = df_pd[initial_mask].copy()

source = ColumnDataSource(df_filtered)
source_full = ColumnDataSource(df_pd)

# Store available dates
available_dates_source = ColumnDataSource({'dates': available_dates})

# Create grid of plots
plots = []
plot_width = 250
plot_height = 250

# Calculate variable bounds
var_bounds = {}
for var in variables:
    var_min = float(df_pd[var].min())
    var_max = float(df_pd[var].max())
    padding = (var_max - var_min) * 0.05
    var_bounds[var] = (var_min - padding, var_max + padding)

# PRE-CREATE shared Range1d objects for each variable
# This ensures all plots for a given variable share the EXACT SAME range object
shared_ranges = {}
for idx, var in enumerate(variables):
    shared_ranges[idx] = Range1d(start=var_bounds[var][0], end=var_bounds[var][1])

# Store histogram sources
hist_sources = {}

for i in range(n_vars):
    row_plots = []
    for j in range(n_vars):
        if j > i:
            row_plots.append(None)
            continue
        
        if i == j:
            # Histograms - use the shared range for x-axis
            n_bins = 100
            hist, edges = np.histogram(df_filtered[variables[i]], bins=n_bins)
            
            hist_data = {
                'top': hist,
                'left': edges[:-1],
                'right': edges[1:],
                'bottom': [0] * len(hist)
            }
            hist_source = ColumnDataSource(hist_data)
            hist_sources[i] = hist_source
            
            p = figure(width=plot_width, height=plot_height,
                      toolbar_location='above',
                      tools='pan,wheel_zoom,box_zoom,reset,save',
                      x_range=shared_ranges[i])  # Use pre-created shared range
            
            p.quad(top='top', bottom='bottom', left='left', right='right',
                  source=hist_source,
                  fill_color='skyblue', line_color='black', alpha=0.7)
            
            p.yaxis.axis_label = 'Count'
            p.yaxis.axis_label_standoff = 10
            
            if i == n_vars - 1:
                p.xaxis.axis_label = labels[j]
            else:
                p.xaxis.major_label_text_font_size = '0pt'
            
        else:
            # Scatter plots - use shared ranges for both axes
            p = figure(width=plot_width, height=plot_height,
                      toolbar_location='above',
                      tools='pan,wheel_zoom,box_zoom,reset,save,hover',
                      x_range=shared_ranges[j],  # Share x-range with column j
                      y_range=shared_ranges[i])  # Share y-range with row i
            
            p.scatter(variables[j], variables[i], source=source,
                     size=5, color='color', alpha=0.6, marker='circle')
            
            hover = p.select_one(HoverTool)
            hover.tooltips = [
                ('Band', '@band'),
                ('day_obs', '@day_obs'),
                (labels[j], f'@{{{variables[j]}}}{{0.000}}'),
                (labels[i], f'@{{{variables[i]}}}{{0.000}}'),
            ]
            
            if i == n_vars - 1:
                p.xaxis.axis_label = labels[j]
            else:
                p.xaxis.major_label_text_font_size = '0pt'
            
            if j == 0:
                p.yaxis.axis_label = labels[i]
            else:
                p.yaxis.major_label_text_font_size = '0pt'
        
        row_plots.append(p)
    
    plots.append(row_plots)

# Create DATE RANGE sliders using indices into available_dates array
date_slider_min = Slider(
    start=0,
    end=n_dates - 1,
    value=start_idx,
    step=1,
    title="Start Date Index",
    width=300
)

date_slider_max = Slider(
    start=0,
    end=n_dates - 1,
    value=end_idx,
    step=1,
    title="End Date Index",
    width=300
)

# Display actual dates
date_display_min = Div(text=f"<strong>{available_dates[start_idx]}</strong>", width=120, height=30)
date_display_max = Div(text=f"<strong>{available_dates[end_idx]}</strong>", width=120, height=30)

# Text inputs for direct date entry
date_text_min = TextInput(value=str(available_dates[start_idx]), title="Or enter date:", width=120)
date_text_max = TextInput(value=str(available_dates[end_idx]), title="Or enter date:", width=120)

# Status
date_status = Div(
    text=f"<p style='margin: 5px 0; color: #666;'>Showing {len(df_filtered)} of {len(df_pd)} points | {end_idx - start_idx + 1} of {n_dates} dates</p>", 
    width=700, 
    height=30
)

# JavaScript callback for sliders
slider_callback = CustomJS(
    args=dict(
        source=source,
        source_full=source_full,
        slider_min=date_slider_min,
        slider_max=date_slider_max,
        display_min=date_display_min,
        display_max=date_display_max,
        text_min=date_text_min,
        text_max=date_text_max,
        date_status=date_status,
        hist_sources=hist_sources,
        variables=variables,
        available_dates=available_dates_source
    ),
    code="""
    const idx_min = Math.floor(slider_min.value);
    const idx_max = Math.floor(slider_max.value);
    
    if (idx_min > idx_max) {
        return;
    }
    
    const dates = available_dates.data['dates'];
    const min_date = dates[idx_min];
    const max_date = dates[idx_max];
    
    display_min.text = '<strong>' + min_date + '</strong>';
    display_max.text = '<strong>' + max_date + '</strong>';
    text_min.value = min_date.toString();
    text_max.value = max_date.toString();
    
    const full_data = source_full.data;
    const day_obs = full_data['day_obs'];
    const filtered_indices = [];
    
    for (let i = 0; i < day_obs.length; i++) {
        if (day_obs[i] >= min_date && day_obs[i] <= max_date) {
            filtered_indices.push(i);
        }
    }
    
    const new_data = {};
    for (const key in full_data) {
        new_data[key] = filtered_indices.map(i => full_data[key][i]);
    }
    source.data = new_data;
    
    const n_dates_in_range = idx_max - idx_min + 1;
    date_status.text = '<p style="margin: 5px 0; color: #666;">Showing ' + filtered_indices.length + 
                      ' of ' + day_obs.length + ' points | ' + n_dates_in_range + ' of ' + dates.length + ' dates</p>';
    
    for (let var_idx = 0; var_idx < variables.length; var_idx++) {
        if (hist_sources[var_idx]) {
            const var_name = variables[var_idx];
            const var_data = new_data[var_name];
            
            if (var_data.length === 0) continue;
            
            const var_min = Math.min(...var_data);
            const var_max = Math.max(...var_data);
            
            const n_bins = 20;
            const bin_width = (var_max - var_min) / n_bins;
            const bins = new Array(n_bins).fill(0);
            const edges_left = [];
            const edges_right = [];
            
            for (let i = 0; i < n_bins; i++) {
                edges_left.push(var_min + i * bin_width);
                edges_right.push(var_min + (i + 1) * bin_width);
            }
            
            for (let i = 0; i < var_data.length; i++) {
                const val = var_data[i];
                if (!isNaN(val)) {
                    const bin_idx = Math.min(Math.floor((val - var_min) / bin_width), n_bins - 1);
                    if (bin_idx >= 0 && bin_idx < n_bins) {
                        bins[bin_idx]++;
                    }
                }
            }
            
            hist_sources[var_idx].data = {
                'top': bins,
                'left': edges_left,
                'right': edges_right,
                'bottom': new Array(n_bins).fill(0)
            };
        }
    }
"""
)

# Callback for text inputs
text_callback = CustomJS(
    args=dict(
        source=source,
        source_full=source_full,
        slider_min=date_slider_min,
        slider_max=date_slider_max,
        display_min=date_display_min,
        display_max=date_display_max,
        text_min=date_text_min,
        text_max=date_text_max,
        date_status=date_status,
        hist_sources=hist_sources,
        variables=variables,
        available_dates=available_dates_source
    ),
    code="""
    const target_min = parseInt(text_min.value);
    const target_max = parseInt(text_max.value);
    
    if (isNaN(target_min) || isNaN(target_max)) {
        date_status.text = '<p style="margin: 5px 0; color: #dc3545;">⚠️ Invalid date format</p>';
        return;
    }
    
    const dates = available_dates.data['dates'];
    
    let idx_min = 0;
    let idx_max = dates.length - 1;
    
    for (let i = 0; i < dates.length; i++) {
        if (dates[i] >= target_min) {
            idx_min = i;
            break;
        }
    }
    
    for (let i = dates.length - 1; i >= 0; i--) {
        if (dates[i] <= target_max) {
            idx_max = i;
            break;
        }
    }
    
    if (idx_min > idx_max) {
        date_status.text = '<p style="margin: 5px 0; color: #dc3545;">⚠️ No data in this range</p>';
        return;
    }
    
    slider_min.value = idx_min;
    slider_max.value = idx_max;
    
    const min_date = dates[idx_min];
    const max_date = dates[idx_max];
    
    display_min.text = '<strong>' + min_date + '</strong>';
    display_max.text = '<strong>' + max_date + '</strong>';
    text_min.value = min_date.toString();
    text_max.value = max_date.toString();
    
    const full_data = source_full.data;
    const day_obs = full_data['day_obs'];
    const filtered_indices = [];
    
    for (let i = 0; i < day_obs.length; i++) {
        if (day_obs[i] >= min_date && day_obs[i] <= max_date) {
            filtered_indices.push(i);
        }
    }
    
    const new_data = {};
    for (const key in full_data) {
        new_data[key] = filtered_indices.map(i => full_data[key][i]);
    }
    source.data = new_data;
    
    const n_dates_in_range = idx_max - idx_min + 1;
    date_status.text = '<p style="margin: 5px 0; color: #28a745;">✓ Showing ' + filtered_indices.length + 
                      ' of ' + day_obs.length + ' points | ' + n_dates_in_range + ' of ' + dates.length + ' dates</p>';
    
    for (let var_idx = 0; var_idx < variables.length; var_idx++) {
        if (hist_sources[var_idx]) {
            const var_name = variables[var_idx];
            const var_data = new_data[var_name];
            
            if (var_data.length === 0) continue;
            
            const var_min = Math.min(...var_data);
            const var_max = Math.max(...var_data);
            
            const n_bins = 20;
            const bin_width = (var_max - var_min) / n_bins;
            const bins = new Array(n_bins).fill(0);
            const edges_left = [];
            const edges_right = [];
            
            for (let i = 0; i < n_bins; i++) {
                edges_left.push(var_min + i * bin_width);
                edges_right.push(var_min + (i + 1) * bin_width);
            }
            
            for (let i = 0; i < var_data.length; i++) {
                const val = var_data[i];
                if (!isNaN(val)) {
                    const bin_idx = Math.min(Math.floor((val - var_min) / bin_width), n_bins - 1);
                    if (bin_idx >= 0 && bin_idx < n_bins) {
                        bins[bin_idx]++;
                    }
                }
            }
            
            hist_sources[var_idx].data = {
                'top': bins,
                'left': edges_left,
                'right': edges_right,
                'bottom': new Array(n_bins).fill(0)
            };
        }
    }
"""
)

date_slider_min.js_on_change('value', slider_callback)
date_slider_max.js_on_change('value', slider_callback)
date_text_min.js_on_change('value', text_callback)
date_text_max.js_on_change('value', text_callback)

# Date controls layout
date_controls = column(
    row(
        column(date_slider_min, date_display_min, date_text_min),
        column(date_slider_max, date_display_max, date_text_max)
    ),
    date_status
)

# Variable sliders - now using shared_ranges
from bokeh.models import RangeSlider

slider_rows = []
for idx, (var, label) in enumerate(zip(variables, labels)):
    var_min, var_max = var_bounds[var]
    
    slider = RangeSlider(
        start=var_min,
        end=var_max,
        value=(var_min, var_max),
        step=(var_max - var_min) / 100,
        title=label,
        width=250
    )
    
    text_min = TextInput(value=f"{var_min:.4f}", title="Min:", width=100)
    text_max = TextInput(value=f"{var_max:.4f}", title="Max:", width=100)
    
    # Now callback updates the shared range directly
    callback_args = {
        'shared_range': shared_ranges[idx],  # Use the shared range object
        'slider': slider,
        'text_min': text_min,
        'text_max': text_max,
        'source': source,
        'var_name': var
    }
    
    if idx in hist_sources:
        callback_args['hist_source'] = hist_sources[idx]
        
        slider_callback_var = CustomJS(args=callback_args, code="""
            // Update the shared range - this affects ALL plots using this variable
            shared_range.start = slider.value[0];
            shared_range.end = slider.value[1];
            
            text_min.value = slider.value[0].toFixed(4);
            text_max.value = slider.value[1].toFixed(4);
            
            const data = source.data;
            const var_data = data[var_name];
            const min_val = slider.value[0];
            const max_val = slider.value[1];
            
            const filtered = [];
            for (let i = 0; i < var_data.length; i++) {
                if (var_data[i] >= min_val && var_data[i] <= max_val) {
                    filtered.push(var_data[i]);
                }
            }
            
            const n_bins = 20;
            const bin_width = (max_val - min_val) / n_bins;
            const bins = new Array(n_bins).fill(0);
            const edges_left = [];
            const edges_right = [];
            
            for (let i = 0; i < n_bins; i++) {
                edges_left.push(min_val + i * bin_width);
                edges_right.push(min_val + (i + 1) * bin_width);
            }
            
            for (let i = 0; i < filtered.length; i++) {
                const val = filtered[i];
                const bin_idx = Math.min(Math.floor((val - min_val) / bin_width), n_bins - 1);
                if (bin_idx >= 0 && bin_idx < n_bins) {
                    bins[bin_idx]++;
                }
            }
            
            hist_source.data['top'] = bins;
            hist_source.data['left'] = edges_left;
            hist_source.data['right'] = edges_right;
            hist_source.data['bottom'] = new Array(n_bins).fill(0);
            hist_source.change.emit();
        """)
        
        text_callback_var = CustomJS(args=callback_args, code="""
            const min_val = parseFloat(text_min.value);
            const max_val = parseFloat(text_max.value);
            
            if (isNaN(min_val) || isNaN(max_val) || min_val >= max_val) {
                return;
            }
            
            slider.value = [min_val, max_val];
            
            // Update the shared range
            shared_range.start = min_val;
            shared_range.end = max_val;
            
            const data = source.data;
            const var_data = data[var_name];
            
            const filtered = [];
            for (let i = 0; i < var_data.length; i++) {
                if (var_data[i] >= min_val && var_data[i] <= max_val) {
                    filtered.push(var_data[i]);
                }
            }
            
            const n_bins = 20;
            const bin_width = (max_val - min_val) / n_bins;
            const bins = new Array(n_bins).fill(0);
            const edges_left = [];
            const edges_right = [];
            
            for (let i = 0; i < n_bins; i++) {
                edges_left.push(min_val + i * bin_width);
                edges_right.push(min_val + (i + 1) * bin_width);
            }
            
            for (let i = 0; i < filtered.length; i++) {
                const val = filtered[i];
                const bin_idx = Math.min(Math.floor((val - min_val) / bin_width), n_bins - 1);
                if (bin_idx >= 0 && bin_idx < n_bins) {
                    bins[bin_idx]++;
                }
            }
            
            hist_source.data['top'] = bins;
            hist_source.data['left'] = edges_left;
            hist_source.data['right'] = edges_right;
            hist_source.data['bottom'] = new Array(n_bins).fill(0);
            hist_source.change.emit();
        """)
    else:
        slider_callback_var = CustomJS(args=callback_args, code="""
            shared_range.start = slider.value[0];
            shared_range.end = slider.value[1];
            text_min.value = slider.value[0].toFixed(4);
            text_max.value = slider.value[1].toFixed(4);
        """)
        
        text_callback_var = CustomJS(args=callback_args, code="""
            const min_val = parseFloat(text_min.value);
            const max_val = parseFloat(text_max.value);
            
            if (isNaN(min_val) || isNaN(max_val) || min_val >= max_val) {
                return;
            }
            
            slider.value = [min_val, max_val];
            shared_range.start = min_val;
            shared_range.end = max_val;
        """)
    
    slider.js_on_change('value', slider_callback_var)
    text_min.js_on_change('value', text_callback_var)
    text_max.js_on_change('value', text_callback_var)
    
    control_row = row(slider, text_min, text_max)
    slider_rows.append(control_row)

sliders_panel = column(*slider_rows)

# Title
unique_bands = sorted(set(bands))
legend_html = ' '.join([f'<span style="color:{band_colors[band]};">●</span> {band}' 
                        for band in unique_bands])

title_div = Div(text=f"""
<div style="text-align: center; font-family: Arial, sans-serif;">
    <h2 style="margin-bottom: 5px;">{science_program}: {test_title}</h2>
    <p style="margin-top: 5px; margin-bottom: 10px;">Available dates: {available_dates[0]} - {available_dates[-1]} ({n_dates} dates)</p>
    <p style="margin-top: 5px;"><strong>Band:</strong> {legend_html}</p>
</div>
""", width=n_vars * plot_width, height=100)

separator1 = Div(text="<hr style='margin: 20px 0; border: none; border-top: 2px solid #667eea;'>", 
                 width=n_vars * plot_width)
separator2 = Div(text="<hr style='margin: 20px 0; border: none; border-top: 1px solid #dee2e6;'>", 
                 width=n_vars * plot_width)

grid = gridplot(plots, toolbar_location='right')

layout = column(
    title_div, 
    separator1,
    date_controls,
    separator2,
    sliders_panel, 
    grid
)

show(layout)

print(f"Title: {science_program}: {test_title}")
print(f"Available dates: {n_dates}")
print(f"Date range: {available_dates[0]} - {available_dates[-1]}")
print(f"Bands: {', '.join(unique_bands)}")